In [15]:
# STEP 1 - Generate an External Knowledge base (python.pdf)

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("C://knowledge_base//Bhagavad-gita_As_It_Is english.pdf")
pages = loader.load()

print("Metadata: \n", pages[0].metadata)

Metadata: 
 {'producer': 'Acrobat Distiller 5.0 (Windows)', 'creator': 'PScript5.dll Version 5.2', 'creationdate': '2002-12-31T18:29:21+00:00', 'moddate': '2007-05-08T20:39:53-04:00', 'title': 'Bhagavad-Gita As It Is', 'author': 'His Divine Grace AC Bhaktivedanta Swami Prabhupada', 'keywords': 'Bhagavad Gita, Krsna, Yoga, Bhakti Yoga, Krishna, Vedas, Hinduism', 'subject': 'Vedic Philosophy', 'source': 'C://knowledge_base//Bhagavad-gita_As_It_Is english.pdf', 'total_pages': 1051, 'page': 0, 'page_label': '1'}


In [16]:
# STEP 2 - Split the knowledge base into documents

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", ",", " "]
)
split_docs = splitter.split_documents(pages)

print(f"Total chunks: ", {len(split_docs)})

Total chunks:  {7403}


In [17]:
# STEP 3 - Create embeddings and vector store using FAISS

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
import os
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv()
if not env_path:
    raise FileNotFoundError(".env file not found.")

load_dotenv(env_path)
apiKey = os.getenv("OPENAI_API_KEY")

embedding_model = OpenAIEmbeddings(model="text-embedding-ada-002", api_key=apiKey)

vectors = FAISS.from_documents(split_docs, embedding=embedding_model)
print(vectors)

In [18]:
# STEP 3 - Create embeddings and vector store using Chroma db

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
import os
from dotenv import load_dotenv, find_dotenv

env_path = find_dotenv()
if not env_path:
    raise FileNotFoundError(".env file not found.")

load_dotenv(env_path)
apiKey = os.getenv("OPENAI_API_KEY")

embedding_model = OpenAIEmbeddings(model="text-embedding-ada-002", api_key=apiKey)

persistent_directory = "c:/content/chroma_db"

vectors = Chroma.from_documents(documents=split_docs,
                                     embedding=embedding_model,
                                     persist_directory=persistent_directory)
print(type(vectors))
print(vectors)

<class 'langchain_community.vectorstores.chroma.Chroma'>


In [19]:
# STEP 4 - Integrate vector database with LLM using Retrieval chain mechanism

from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_classic.chains import RetrievalQA

custom_prompt_template = """
Use the following pieces of context to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
--------------------------
{context}
Question: {question}
"""
CUSTOM_PROMPT = PromptTemplate(
    template=custom_prompt_template,
    input_variables=["context", "question"]
)
llm = ChatOpenAI(model="gpt-4o-mini")
qa_chain = RetrievalQA.from_chain_type(llm,
    chain_type="stuff",
    retriever=vectors.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={"prompt": CUSTOM_PROMPT}
)

In [20]:
# STEP 5 - Build CLI based interface

# Loop to ask multiple questions
while True:
    question = input("\nEnter the question (or type 'exit' to quit): ")

    if question.lower() in ['exit', 'quit']:
        print("Exiting... Have a great day!")
        break

    response = qa_chain({"query": question})
    answer = response["result"]
    source_documents = response["source_documents"]

    # Show the answer
    print("\nAnswer:", answer)
    print(f"Source documents: {source_documents}")


Answer: The Bhagavad-gita is a conversation between Kåñëa and Arjuna.
Source documents: [Document(metadata={'title': 'Bhagavad-Gita As It Is', 'total_pages': 1051, 'keywords': 'Bhagavad Gita, Krsna, Yoga, Bhakti Yoga, Krishna, Vedas, Hinduism', 'subject': 'Vedic Philosophy', 'moddate': '2007-05-08T20:39:53-04:00', 'author': 'His Divine Grace AC Bhaktivedanta Swami Prabhupada', 'page': 1026, 'source': 'C://Anand//old_laptop_backup//D drive data//Anand//material//Training//Gen_AI//L2_Applied_Gen_AI//external_knowledgebase_for_rag//Bhagavad-gita_As_It_Is english.pdf', 'creator': 'PScript5.dll Version 5.2', 'producer': 'Acrobat Distiller 5.0 (Windows)', 'page_label': '1027', 'creationdate': '2002-12-31T18:29:21+00:00'}, page_content='discussion of topics between two friends on a battlefield. But such a book\ncannot be scripture. Some may protest that Kåñëa incited Arjuna to fight,\nwhich is immoral, but the reality of  the situation is clearly stated:\nBhagavad-gétä is the supreme instruc

In [21]:
# STEP 6 - User interface

import gradio as gr

def chatbot_response(message, history):
    # Use the created RetrievalQA chain to get the answer
    response = qa_chain({"query": message})
    answer = response["result"]
    source_documents = response["source_documents"]

    # Format the response to include the answer and source documents (optional)
    formatted_response = f"{answer}" # You can add source documents here if desired

    return formatted_response

# Create the Gradio interface
iface = gr.ChatInterface(
    fn=chatbot_response,
    title="RAG Chatbot",
    description="Ask questions about Gen-AI and Langchain based on the provided text."
)

# Launch the interface
iface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Missing file: C:\Users\ak60492\.cache\huggingface\gradio\frpc\frpc_windows_amd64_v0.3. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_windows_amd64.exe
2. Rename the downloaded file to: frpc_windows_amd64_v0.3
3. Move the file to this location: C:\Users\ak60492\.cache\huggingface\gradio\frpc
